# Étape 16 — Deuxième LLM : Llama-3.1-8B-Instruct

**Objectif.** Vérifier que nos conclusions sur les graphes générés par LLM ne dépendent pas de **Qwen2.5-7B**.
On interroge un modèle d'une **autre famille**, **Llama-3.1-8B-Instruct** (Meta), avec **exactement le même protocole** : les mêmes 22 760 paires candidates (même thème), la même consigne, la même mesure (probabilité de « Yes »), la même quantification 4 bits.

**Ce qu'on compare entre les deux LLM :**
1. **Qualité (RQ1)** : AUC face aux prérequis experts (directs, et directs ou indirects), précision sur les |E*| meilleures paires, nature des erreurs (flèches inversées, sans rapport…), sur-confiance, accord entre les deux LLM ;
2. **Concentration et couverture** du graphe « |E*| meilleures paires » (le défaut identifié à l'étape 11) ;
3. **Utilité pour le KT (RQ2)** : graphes « k meilleurs prérequis par exercice » (k = 1, 2, 3, 5), PrereqKT, 5 graines, 5 % et 1 % des élèves, comparés au thème aléatoire et à Qwen (résultats de l'étape 12, mêmes graines).

### Avant de lancer (une seule fois)
1. Sur https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct, connectez-vous et **acceptez la licence** de Meta ; attendez l'e-mail d'accord.
2. Sur https://huggingface.co/settings/tokens, créez un **jeton** (type *Read*).
3. Dans Colab : icône **clé** (« Secrets ») à gauche → ajoutez un secret nommé **`HF_TOKEN`** avec votre jeton, et activez **« Accès au notebook »**.

**GPU T4 obligatoire.** Durée : environ 3 à 5 h pour les 22 760 paires (cellule E1, reprise automatique en cas de déconnexion), puis environ 30 min d'entraînements (E3).

In [ ]:
# Cellule 1 — Drive, GPU, paramètres
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, random, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score, accuracy_score
from scipy import stats

DOSSIER   = '/content/drive/MyDrive/projet_kt'
PREP      = os.path.join(DOSSIER, 'junyi_prepare')
RESULTATS = os.path.join(DOSSIER, 'resultats')
os.makedirs(RESULTATS, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Appareil :', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '(pas de GPU : ce sera lent)')

HP = dict(dim=64, hidden=128, couches_graphe=2, dropout=0.2, lr=1e-3, batch=64, epoques_max=60, patience=5)
GRAINE = 42

In [ ]:
# Cellule 2 — Séquences, graphe expert, et repérage des PREMIÈRES tentatives
exos = pd.read_csv(os.path.join(PREP, 'exercices.csv'))
N = len(exos)
A_expert = np.load(os.path.join(PREP, 'adjacence_expert.npy')).astype(np.float32)
aretes_expert = list(zip(*np.nonzero(A_expert)))

def charger(partie):
    df = pd.read_csv(os.path.join(PREP, f'sequences_{partie}.csv')).sort_values(['uid', 'morceau'])
    U, Q, R, F = [], [], [], []
    vus = {}
    for uid, ex, rep in zip(df['uid'], df['exercices'], df['reponses']):
        q = np.array(ex.split(','), dtype=np.int64)
        r = np.array(rep.split(','), dtype=np.int64)
        deja = vus.setdefault(uid, set())          # exercices déjà tentés par cet élève (tous morceaux confondus)
        f = np.zeros(len(q), dtype=bool)
        for i, x in enumerate(q):
            if x not in deja:
                f[i] = True
                deja.add(x)
        U.append(uid); Q.append(q); R.append(r); F.append(f)
    return np.array(U), Q, R, F

donnees = {p: charger(p) for p in ['train', 'valid', 'test']}
for p, (u, q, r, f) in donnees.items():
    n_rep = sum(map(len, q)); n_first = sum(int(x.sum()) for x in f)
    print(f'{p:5s} : {len(set(u)):6,} élèves | {n_rep:9,} réponses | premières tentatives : {n_first:8,} ({n_first/n_rep:.1%})')

In [ ]:
# Cellule 3 — Tenseurs (Q exercices, R réponses, M masque, F première tentative)
def tenseurs(q_list, r_list, f_list):
    L = max(map(len, q_list))
    Q = np.zeros((len(q_list), L), dtype=np.int64); R = np.zeros_like(Q)
    M = np.zeros((len(q_list), L), dtype=bool);     F = np.zeros_like(M)
    for i, (q, r, f) in enumerate(zip(q_list, r_list, f_list)):
        Q[i, :len(q)], R[i, :len(r)], M[i, :len(q)], F[i, :len(f)] = q, r, True, f
    return tuple(torch.from_numpy(x) for x in (Q, R, M, F))

def sous_ensemble_train(fraction, graine=GRAINE):
    u, q, r, f = donnees['train']
    if fraction >= 1:
        return q, r, f
    rng = np.random.default_rng(graine)
    eleves = np.unique(u)
    gardes = set(rng.choice(eleves, size=max(1, int(fraction * len(eleves))), replace=False))
    idx = [i for i, x in enumerate(u) if x in gardes]
    return [q[i] for i in idx], [r[i] for i in idx], [f[i] for i in idx]

T_valid = tenseurs(*donnees['valid'][1:])
T_test  = tenseurs(*donnees['test'][1:])
print('OK')

### Les modèles

- **GraphKT** : identique à l'étape 5 (le graphe lisse les représentations des exercices, avec un canal « prérequis » et un canal « successeurs »).
- **PrereqKT** : GraphKT **plus** 6 indicateurs calculés à chaque instant pour l'exercice à prédire :
  taux de réussite moyen de l'élève sur ses **prérequis** déjà tentés, part des prérequis déjà tentés, présence de prérequis, et les trois mêmes indicateurs pour ses **successeurs**.
  C'est la traduction directe de l'hypothèse « *s'il a échoué sur A, il échouera plus souvent sur B* ».
  Avec un graphe vide, ces indicateurs valent 0 : PrereqKT redevient GraphKT sans graphe.

In [ ]:
# Cellule 4 — Modèles
def normaliser_lignes(A):
    deg = A.sum(1, keepdims=True)
    return np.divide(A, deg, out=np.zeros_like(A), where=deg > 0)

class GraphKT(nn.Module):
    def __init__(self, n, hp, A, indicateurs=False):
        super().__init__()
        d, h = hp['dim'], hp['hidden']
        A = np.asarray(A, dtype=np.float32)
        self.register_buffer('P_pre', torch.from_numpy(normaliser_lignes(A.T.copy())))  # ligne j : prérequis de j
        self.register_buffer('P_suc', torch.from_numpy(normaliser_lignes(A.copy())))    # ligne i : successeurs de i
        self.n, self.indicateurs = n, indicateurs
        self.E = nn.Embedding(n, d)
        self.couches = nn.ModuleList([nn.ModuleDict({'pre': nn.Linear(d, d), 'suc': nn.Linear(d, d)})
                                      for _ in range(hp['couches_graphe'])])
        self.R = nn.Embedding(2, d)
        self.lstm = nn.LSTM(2 * d, h, batch_first=True)
        self.drop = nn.Dropout(hp['dropout'])
        n_ind = 6 if indicateurs else 0
        self.pred = nn.Sequential(nn.Linear(h + d + n_ind, h), nn.ReLU(), nn.Linear(h, 1))

    def representations(self):
        H = self.E.weight
        a_pre = (self.P_pre.sum(1, keepdim=True) > 0).float()
        a_suc = (self.P_suc.sum(1, keepdim=True) > 0).float()
        for c in self.couches:
            H = H + torch.relu(c['pre'](self.P_pre @ H)) * a_pre + torch.relu(c['suc'](self.P_suc @ H)) * a_suc
        return H

    def calcul_indicateurs(self, q, r):
        # Historique cumulé par exercice jusqu'à l'instant t : nombre d'essais et de réussites (B, L-1, N)
        B, L = q.shape
        essais = torch.zeros(B, L - 1, self.n, device=q.device)
        essais.scatter_(2, q[:, :-1].unsqueeze(-1), 1.0)
        reussites = essais * r[:, :-1].unsqueeze(-1).float()
        essais, reussites = essais.cumsum(1), reussites.cumsum(1)
        tente = (essais > 0).float()
        taux = reussites / essais.clamp(min=1)
        sortie = []
        for P in (self.P_pre, self.P_suc):
            voisins = (P[q[:, 1:]] > 0).float()                 # voisins de l'exercice à prédire (B, L-1, N)
            n_vois = voisins.sum(-1)
            n_tentes = (voisins * tente).sum(-1)
            moy = (voisins * tente * taux).sum(-1) / n_tentes.clamp(min=1)
            sortie += [moy, n_tentes / n_vois.clamp(min=1), (n_vois > 0).float()]
        return torch.stack(sortie, -1)

    def forward(self, q, r):
        H = self.representations()
        h, _ = self.lstm(torch.cat([H[q[:, :-1]], self.R(r[:, :-1])], dim=-1))
        x = [self.drop(h), H[q[:, 1:]]]
        if self.indicateurs:
            x.append(self.calcul_indicateurs(q, r))
        return self.pred(torch.cat(x, dim=-1)).squeeze(-1)

print('OK')

In [ ]:
# Cellule 5 — Graphes : expert, aucun, aléatoire, et corruption contrôlée E1–E5
import networkx as nx

def vers_matrice(aretes, n=N):
    A = np.zeros((n, n), dtype=np.float32)
    for a, b in aretes:
        A[a, b] = 1
    return A

def corrompre(aretes, type_err, taux, graine=GRAINE, topics=exos['topic'].values):
    rng = random.Random(10_000 + graine)
    E = set(map(tuple, aretes))
    k = int(round(taux * len(E)))
    noeuds = list(range(N))
    if type_err == 'E1':      # fausses flèches aléatoires
        while k > 0:
            a, b = rng.sample(noeuds, 2)
            if (a, b) not in E and (b, a) not in E:
                E.add((a, b)); k -= 1
    elif type_err == 'E2':    # fausses flèches plausibles : même thème
        par_topic = {}
        for i, t in enumerate(topics):
            par_topic.setdefault(t, []).append(i)
        groupes = [g for g in par_topic.values() if len(g) >= 2]
        essais = 0
        while k > 0 and essais < 100 * len(E) + 1000:
            essais += 1
            a, b = rng.sample(rng.choice(groupes), 2)
            if (a, b) not in E and (b, a) not in E:
                E.add((a, b)); k -= 1
    elif type_err == 'E3':    # inversions
        for a, b in rng.sample(sorted(E), min(k, len(E))):
            E.discard((a, b)); E.add((b, a))
    elif type_err == 'E4':    # suppressions
        for e in rng.sample(sorted(E), min(k, len(E))):
            E.discard(e)
    elif type_err == 'E5':    # cycles : C -> A alors que A mène déjà à C
        G = nx.DiGraph(list(E))
        candidats = sorted(E); rng.shuffle(candidats)
        for a, b in candidats:
            if k == 0:
                break
            c = rng.choice(list(nx.descendants(G, b)) + [b])
            if c != a and (c, a) not in E:
                E.add((c, a)); G.add_edge(c, a); k -= 1
    return sorted(E)

def graphe(nom, graine=GRAINE, taux=0.0):
    if nom == 'expert':
        return A_expert
    if nom == 'aucun':
        return np.zeros((N, N), dtype=np.float32)
    if nom == 'aleatoire':
        rng = random.Random(graine)
        E = set()
        while len(E) < len(aretes_expert):
            a, b = rng.sample(range(N), 2)
            E.add((a, b))
        return vers_matrice(E)
    return vers_matrice(corrompre(aretes_expert, nom, taux, graine))

def f1_aretes(A, B=A_expert):
    P, V = set(zip(*np.nonzero(A))), set(zip(*np.nonzero(B)))
    tp = len(P & V); p = tp / max(1, len(P)); r = tp / max(1, len(V))
    return round(2 * p * r / (p + r), 3) if p + r else 0.0

print(f'{"graphe":8s} {"taux":>5s} {"flèches":>8s} {"F1/expert":>10s} {"acyclique":>10s}')
for t in ['E1', 'E2', 'E3', 'E4', 'E5']:
    for taux in [0.1, 0.3, 0.5]:
        A = graphe(t, 0, taux)
        print(f'{t:8s} {taux:5.0%} {int(A.sum()):8d} {f1_aretes(A):10.3f} {str(nx.is_directed_acyclic_graph(nx.DiGraph(A))):>10s}')

In [ ]:
# Cellule 6 — Entraînement et évaluation par sous-ensemble
def fixer_graine(g):
    random.seed(g); np.random.seed(g); torch.manual_seed(g); torch.cuda.manual_seed_all(g)

@torch.no_grad()
def evaluer(modele, T, batch=128):
    modele.eval()
    Q, R, M, F = T
    ys, ps, fs = [], [], []
    for i in range(0, len(Q), batch):
        q, r, m, f = (x[i:i+batch].to(DEVICE) for x in (Q, R, M, F))
        p = torch.sigmoid(modele(q, r))
        c = m[:, 1:]
        ys.append(r[:, 1:][c].cpu().numpy()); ps.append(p[c].float().cpu().numpy()); fs.append(f[:, 1:][c].cpu().numpy())
    y, p, f = (np.concatenate(x) for x in (ys, ps, fs))
    auc = lambda s: round(float(roc_auc_score(y[s], p[s])), 4) if len(set(y[s])) == 2 else None
    tout = np.ones_like(f, dtype=bool)
    return {'AUC': auc(tout), 'AUC_1re': auc(f), 'AUC_rep': auc(~f), 'ACC': round(float(accuracy_score(y, p >= 0.5)), 4)}

def entrainer(nom_modele, A, fraction=1.0, graine=GRAINE, hp=HP):
    fixer_graine(graine)
    Q, R, M, F = tenseurs(*sous_ensemble_train(fraction, graine))
    modele = GraphKT(N, hp, A, indicateurs=(nom_modele == 'PrereqKT')).to(DEVICE)
    opt = torch.optim.Adam(modele.parameters(), lr=hp['lr'])
    perte = nn.BCEWithLogitsLoss()
    meilleur, etat, attente, t0 = -1, None, 0, time.time()
    for ep in range(hp['epoques_max']):
        modele.train()
        ordre = torch.randperm(len(Q))
        for i in range(0, len(Q), hp['batch']):
            idx = ordre[i:i + hp['batch']]
            q, r, m = Q[idx].to(DEVICE), R[idx].to(DEVICE), M[idx].to(DEVICE)
            c = m[:, 1:]
            l = perte(modele(q, r)[c], r[:, 1:][c].float())
            opt.zero_grad(); l.backward()
            nn.utils.clip_grad_norm_(modele.parameters(), 5.0)
            opt.step()
        v = evaluer(modele, T_valid)['AUC']
        # print(f'   époque {ep+1:2d} | AUC valid {v:.4f} | {(time.time()-t0)/60:.1f} min')
        if v > meilleur + 1e-4:
            meilleur, etat, attente = v, copy.deepcopy(modele.state_dict()), 0
        else:
            attente += 1
            if attente >= hp['patience']:
                break
    modele.load_state_dict(etat)
    return {'AUC_valid': round(meilleur, 4), **{f'{k}_test': v for k, v in evaluer(modele, T_test).items()},
            'epoques': ep + 1, 'minutes': round((time.time() - t0) / 60, 1)}
print('OK')

In [ ]:
# Cellule C1 — Paires candidates et couverture du graphe expert
# Périmètre : même thème (topic) par défaut ; même domaine (area) si le thème couvre moins de 70 % des flèches expertes
exos['texte'] = exos['name'].str.replace('_', ' ').str.strip()
def paires_par(colonne):
    groupes = exos.dropna(subset=[colonne]).groupby(colonne)['exo_id'].apply(list)
    return [(a, b) for ids in groupes for a in ids for b in ids if a != b]
couverture = {}
for colonne in ['topic', 'area']:
    c = paires_par(colonne)
    cs = set(c)
    couverture[colonne] = (c, sum(1 for e in aretes_expert if tuple(e) in cs) / len(aretes_expert))
    print(f'Périmètre « {colonne} » : {len(c):,} paires | flèches expertes couvertes : {couverture[colonne][1]:.1%}')
PERIMETRE = 'topic' if couverture['topic'][1] >= 0.70 or len(couverture['area'][0]) > 120_000 else 'area'
candidats, rappel_max = couverture[PERIMETRE]
print(f'\n=> Périmètre retenu : « {PERIMETRE} » — {len(candidats):,} paires, rappel maximal {rappel_max:.1%}')
print('Exemples :', [(exos.texte[a], exos.texte[b]) for a, b in candidats[:3]])

In [ ]:
# Cellule E1 — Llama-3.1-8B-Instruct (4 bits) : probabilité de « Yes » pour toutes les paires (reprise automatique)
!pip install -q -U transformers accelerate bitsandbytes
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get('HF_TOKEN'))

MODELE_LLM = 'meta-llama/Llama-3.1-8B-Instruct'
tok = AutoTokenizer.from_pretrained(MODELE_LLM)
tok.padding_side = 'left'
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
llm = AutoModelForCausalLM.from_pretrained(
    MODELE_LLM, device_map='auto',
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16))
llm.eval()
I_OUI = tok.encode('Yes', add_special_tokens=False)[0]
I_NON = tok.encode('No', add_special_tokens=False)[0]
print('Jetons « Yes » / « No » :', I_OUI, I_NON, tok.convert_ids_to_tokens([I_OUI, I_NON]))

# Consigne IDENTIQUE à celle utilisée pour Qwen (étape 9, cellule B2)
SYSTEME = 'You are an expert mathematics teacher who designs curricula.'
GABARIT = ('Two exercises from an online mathematics course (topics: "{topic}" and "{topic_b}"):\n'
           'Exercise A: {a}\nExercise B: {b}\n'
           'Must a student master exercise A before attempting exercise B '
           '(is A a prerequisite of B)? Answer only Yes or No.')

@torch.no_grad()
def proba_oui(paires, taille_lot=32):
    textes = [tok.apply_chat_template(
                  [{'role': 'system', 'content': SYSTEME},
                   {'role': 'user', 'content': GABARIT.format(topic=str(exos.topic[a]).replace('-', ' '), topic_b=str(exos.topic[b]).replace('-', ' '),
                                                               a=exos.texte[a], b=exos.texte[b])}],
                  tokenize=False, add_generation_prompt=True) for a, b in paires]
    sortie = []
    for i in range(0, len(textes), taille_lot):
        lot = tok(textes[i:i + taille_lot], return_tensors='pt', padding=True, add_special_tokens=False).to(llm.device)
        z = llm(**lot).logits[:, -1, :].float()
        sortie += torch.softmax(torch.stack([z[:, I_OUI], z[:, I_NON]], 1), 1)[:, 0].cpu().tolist()
    return sortie

nv = dict(zip(exos.name, exos.exo_id))
if {'factoring_polynomials_1', 'solving_quadratics_by_factoring'} <= set(nv):
    print('Test (vrai prérequis)    :', proba_oui([(nv['factoring_polynomials_1'], nv['solving_quadratics_by_factoring'])]))
    print('Test (sens inversé)      :', proba_oui([(nv['solving_quadratics_by_factoring'], nv['factoring_polynomials_1'])]))

chemin_scores = os.path.join(RESULTATS, 'junyi_llm_scores_llama.csv')
scores_l = pd.read_csv(chemin_scores) if os.path.exists(chemin_scores) else pd.DataFrame(columns=['A', 'B', 'p_oui'])
deja = set(zip(scores_l.A.astype(int), scores_l.B.astype(int)))
reste = [p for p in candidats if p not in deja]
print(f'Déjà évaluées : {len(deja):,} | restantes : {len(reste):,}')
t0, BLOC = time.time(), 2048
for i in range(0, len(reste), BLOC):
    part = reste[i:i + BLOC]
    p = proba_oui(part)
    scores_l = pd.concat([scores_l, pd.DataFrame({'A': [a for a, _ in part], 'B': [b for _, b in part], 'p_oui': p})], ignore_index=True)
    scores_l.to_csv(chemin_scores, index=False)
    fait_ = i + len(part)
    print(f'{fait_:,}/{len(reste):,} paires | {(time.time() - t0) / 60:.1f} min | reste ≈ {(time.time() - t0) / fait_ * (len(reste) - fait_) / 60:.0f} min')
print('Génération terminée :', len(scores_l), 'paires')

In [ ]:
# Cellule E2 — Qualité des deux LLM face au graphe expert (RQ1), accord entre eux, concentration et couverture
#   Après une déconnexion : relancez les cellules 1 à C1, puis celle-ci (pas besoin de E1 si elle est terminée).
import gc
import networkx as nx
if 'llm' in globals():
    del llm; gc.collect(); torch.cuda.empty_cache()

G_exp = nx.DiGraph(); G_exp.add_nodes_from(range(N)); G_exp.add_edges_from(map(tuple, aretes_expert))
desc = {n: nx.descendants(G_exp, n) for n in G_exp}
comp = {}
for i, c in enumerate(nx.weakly_connected_components(G_exp)):
    for n in c:
        comp[n] = i
E_dir = set(map(tuple, aretes_expert))
def categorie(a, b):
    if (a, b) in E_dir:      return 'direct'
    if b in desc[a]:         return 'indirect'
    if (b, a) in E_dir:      return 'inverse'
    if a in desc[b]:         return 'indirect_inverse'
    if comp[a] == comp[b]:   return 'meme_composante'
    return 'sans_rapport'
ORDRE = ['direct', 'indirect', 'inverse', 'indirect_inverse', 'meme_composante', 'sans_rapport']

def lire_scores(nom):
    s = pd.read_csv(os.path.join(RESULTATS, nom)); s[['A', 'B']] = s[['A', 'B']].astype(int)
    return s.drop_duplicates(['A', 'B']).set_index(['A', 'B'])['p_oui']
SC = {'Qwen2.5-7B': lire_scores('junyi_llm_scores.csv'), 'Llama-3.1-8B': lire_scores('junyi_llm_scores_llama.csv')}
cles = [p for p in candidats if p in SC['Qwen2.5-7B'].index and p in SC['Llama-3.1-8B'].index]
print(f'Paires évaluées par les deux LLM : {len(cles):,} / {len(candidats):,}')
y_dir = np.array([p in E_dir for p in cles]); y_ind = np.array([b in desc[a] for a, b in cles])

_, Qt, _, Ft = donnees['test']
premieres = pd.Series(np.concatenate([q[f] for q, f in zip(Qt, Ft)])).value_counts()
poids = premieres.reindex(range(N), fill_value=0).values
def gini(x):
    x = np.sort(np.asarray(x, float)); n = len(x)
    return float((2 * np.arange(1, n + 1) - n - 1).dot(x) / (n * x.sum())) if x.sum() else 0.0

E_star = len(aretes_expert)
lignes, cats = [], {}
for nom, s in SC.items():
    p = s.loc[cles].values
    alea = np.random.default_rng(123).random(len(p))
    ordre = np.lexsort((alea, -p))                      # tri par p décroissant, ex æquo départagés au hasard (graine fixée)
    top = [cles[i] for i in ordre[:E_star]]
    c = pd.Series([categorie(a, b) for a, b in top]).value_counts(normalize=True).reindex(ORDRE, fill_value=0)
    cats[nom] = (100 * c).round(1)
    deg_out = np.bincount([a for a, _ in top], minlength=N); a_pre = np.bincount([b for _, b in top], minlength=N) > 0
    lignes.append({'LLM': nom, 'AUC_prerequis_directs': round(roc_auc_score(y_dir, p), 4),
                   'AUC_directs_ou_indirects': round(roc_auc_score(y_ind, p), 4),
                   'precision_top_E*_%': round(100 * c['direct'], 1), 'precision_elargie_top_E*_%': round(100 * (c['direct'] + c['indirect']), 1),
                   'inversees_top_E*_%': round(100 * (c['inverse'] + c['indirect_inverse']), 1),
                   'part_p>=0.5_%': round(100 * float((p >= 0.5).mean()), 1), 'part_p>=0.999_%': round(100 * float((p >= 0.999).mean()), 1),
                   'couverture_1res_top_E*_%': round(100 * (poids * a_pre).sum() / poids.sum(), 1),
                   'part_fleches_top10_sources_%': round(100 * np.sort(deg_out)[::-1][:10].sum() / E_star, 1),
                   'gini_degre_sortant': round(gini(deg_out), 3)})
qual2 = pd.DataFrame(lignes)
c_h = pd.Series([categorie(a, b) for a, b in cles]).value_counts(normalize=True).reindex(ORDRE, fill_value=0)
cats['hasard (toutes candidates)'] = (100 * c_h).round(1)
cats = pd.DataFrame(cats).T

pq, pl = SC['Qwen2.5-7B'].loc[cles].values, SC['Llama-3.1-8B'].loc[cles].values
rq = pd.Series(pq).rank(method='first', ascending=False).values <= E_star
rl = pd.Series(pl).rank(method='first', ascending=False).values <= E_star
accord = pd.DataFrame([{'spearman_p_oui': round(stats.spearmanr(pq, pl).correlation, 3),
                        'recouvrement_top_E*_jaccard': round((rq & rl).sum() / (rq | rl).sum(), 3),
                        'accord_decision_p>=0.5_%': round(100 * float(((pq >= 0.5) == (pl >= 0.5)).mean()), 1),
                        'AUC_moyenne_des_deux': round(roc_auc_score(y_dir, (pq + pl) / 2), 4)}])

qual2.to_csv(os.path.join(RESULTATS, 'rq1_deux_llm_qualite.csv'), index=False)
cats.to_csv(os.path.join(RESULTATS, 'rq1_deux_llm_categories.csv'))
accord.to_csv(os.path.join(RESULTATS, 'rq1_deux_llm_accord.csv'), index=False)
print('===== Qualité face au graphe expert =====\n', qual2.T.to_string())
print('\n===== Nature des |E*| flèches les plus probables (%) =====\n', cats.to_string())
print('\n===== Accord entre les deux LLM =====\n', accord.to_string(index=False))

In [ ]:
# Cellule E3 — Utilité pour le KT : Llama top-k par exercice (k = 1, 2, 3, 5), PrereqKT, 5 graines, 5 % et 1 % (reprise automatique)
K_LISTE, GRAINES = [1, 2, 3, 5], [0, 1, 2, 3, 4]
sl = SC['Llama-3.1-8B'].reset_index()
sl['alea'] = np.random.default_rng(123).random(len(sl))
sl = sl.sort_values(['B', 'p_oui', 'alea'], ascending=[True, False, True])
def llama_topk(k):
    sel = sl.groupby('B', sort=False).head(k)
    return list(zip(sel.A.astype(int), sel.B.astype(int)))

chemin_tl = os.path.join(RESULTATS, 'rq2_topk_llama.csv')
res_tl = pd.read_csv(chemin_tl).to_dict('records') if os.path.exists(chemin_tl) else []
fait = {(x['graphe'], float(x['fraction']), int(x['graine'])) for x in res_tl}
t0_ = time.time()
for frac in [0.05, 0.01]:
    for k in K_LISTE:
        lib = f'Llama_k{k}'
        E = llama_topk(k)
        for g_ in GRAINES:
            if (lib, frac, g_) in fait:
                continue
            res = entrainer('PrereqKT', vers_matrice(E), fraction=frac, graine=g_)
            res_tl.append({'modele': 'PrereqKT', 'graphe': lib, 'k': k, 'fraction': frac, 'graine': g_, **res})
            pd.DataFrame(res_tl).to_csv(chemin_tl, index=False)
            print(f'{lib:9s} | {frac:.0%} | graine {g_} -> AUC 1res {res["AUC_1re_test"]} ({(time.time() - t0_) / 60:.0f} min)')
print('Entraînements terminés.')

In [ ]:
# Cellule E4 — Analyse KT : Llama contre thème aléatoire et contre Qwen (mêmes graines, résultats de l'étape 12)
import matplotlib.pyplot as plt
col = 'AUC_1re_test'
ref = pd.read_csv(os.path.join(RESULTATS, 'phase2_confirmation.csv'))
ref = ref[ref.modele == 'PrereqKT'][['graphe', 'fraction', 'graine', col]]
qw = pd.read_csv(os.path.join(RESULTATS, 'rq2_topk.csv'))[['graphe', 'fraction', 'graine', col]]
qw['graphe'] = qw['graphe'].str.replace('LLM_k', 'Qwen_k', regex=False)
ll = pd.read_csv(os.path.join(RESULTATS, 'rq2_topk_llama.csv'))[['graphe', 'fraction', 'graine', col]]
tout = pd.concat([ref, qw, ll], ignore_index=True)
tout = tout[tout.graine.isin(GRAINES)]

lignes, tests = [], []
for frac in [0.05, 0.01]:
    piv = tout[tout.fraction == frac].pivot_table(index='graine', columns='graphe', values=col)
    gain = piv['expert'] - piv['aucun']
    for g in [c for c in piv.columns if c in ('expert', 'aucun') or c[-2:] in ('k1', 'k2', 'k3', 'k5')]:
        lignes.append({'fraction': frac, 'graphe': g, 'AUC_1re': round(piv[g].mean(), 4),
                       'benefice_conserve_%': round(100 * ((piv[g] - piv['aucun']) / gain).mean(), 1)})
    for k in K_LISTE:
        for a, b in [(f'Llama_k{k}', f'theme_k{k}'), (f'Llama_k{k}', f'Qwen_k{k}')]:
            if a in piv and b in piv:
                d = (piv[a] - piv[b]).dropna()
                tests.append({'fraction': frac, 'comparaison': f'{a} − {b}', 'ecart_moyen': round(d.mean(), 4),
                              'gagne': f'{int((d > 0).sum())}/{len(d)}', 'p': stats.ttest_1samp(d, 0).pvalue})
benef = pd.DataFrame(lignes); tests = pd.DataFrame(tests)
o = tests['p'].sort_values(); m = len(o)
tests['p_holm'] = (o * (m - np.arange(m))).cummax().clip(upper=1).reindex(tests.index)
tests['significatif_holm_5%'] = tests['p_holm'] < 0.05
tests[['p', 'p_holm']] = tests[['p', 'p_holm']].round(4)
benef.to_csv(os.path.join(RESULTATS, 'rq2_deux_llm_benefice.csv'), index=False)
tests.to_csv(os.path.join(RESULTATS, 'rq2_deux_llm_tests.csv'), index=False)
for frac in [0.05, 0.01]:
    print(f'\n===== Bénéfice conservé | {frac:.0%} des élèves =====')
    print(benef[benef.fraction == frac].drop(columns='fraction').sort_values('benefice_conserve_%', ascending=False).to_string(index=False))
print(f'\n===== Llama contre thème aléatoire et contre Qwen (bilatéral, Holm sur {m} tests) =====\n', tests.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
for ax, frac in zip(axes, [0.05, 0.01]):
    b = benef[benef.fraction == frac].set_index('graphe')['benefice_conserve_%']
    for fam, st in [('Qwen', 'o-'), ('Llama', '^-'), ('theme', 's--')]:
        ax.plot(K_LISTE, [b.get(f'{fam}_k{k}', np.nan) for k in K_LISTE], st,
                label={'Qwen': 'Qwen2.5-7B', 'Llama': 'Llama-3.1-8B', 'theme': 'thème aléatoire'}[fam])
    ax.axhline(100, ls=':', c='black', lw=1, label='expert')
    ax.set_xticks(K_LISTE); ax.set_xlabel('k (prérequis par exercice)'); ax.set_title(f'{frac:.0%} des élèves')
axes[0].set_ylabel('Bénéfice conservé (%)'); axes[0].legend(fontsize=7)
fig.tight_layout(); fig.savefig(os.path.join(RESULTATS, 'rq2_deux_llm.png'), dpi=200); plt.show()

In [ ]:
# Cellule E5 — Rassembler les résultats de l'étape 16 dans un zip et le télécharger
import zipfile
FICHIERS = ['rq1_deux_llm_qualite.csv', 'rq1_deux_llm_categories.csv', 'rq1_deux_llm_accord.csv',
            'rq2_topk_llama.csv', 'rq2_deux_llm_benefice.csv', 'rq2_deux_llm_tests.csv', 'rq2_deux_llm.png']
chemin_zip = '/content/resultats_etape16.zip'
with zipfile.ZipFile(chemin_zip, 'w') as z:
    for f in FICHIERS:
        c = os.path.join(RESULTATS, f)
        if os.path.exists(c):
            z.write(c, arcname=f); print('ajouté   :', f)
        else:
            print('MANQUANT :', f)
try:
    from google.colab import files
    files.download(chemin_zip)
except ImportError:
    print('Hors de Colab : le fichier est ici ->', chemin_zip)